<h2>Semantic Similarity with FAISS</h2>
<p>
Welcome to a hands-on exploration of semantic search, where we unravel the intricacies of finding meaning in text. This lab is a beginner's journey into the realm of advanced information retrieval. You'll start by learning the essentials of text preprocessing to enhance data quality. Next, you'll dive into the world of vector spaces, using the <u>Universal Sentence Encoder</u> to convert text into a format that machines understand. Finally, you'll harness the efficiency of FAISS, a library built for rapid similarity search, to compare and retrieve information. By the end of our session, you'll have a functional semantic search engine that not only understands the subtleties of human language but also fetches information that truly matters.
<p>

<h2>Objectives</h2>
In this lab, our objectives are to:
<p>
- Understand the fundamentals of semantic search and its advantages over traditional search methods.
- Familiarize with the process of preparing text data for semantic analysis, including cleaning and standardization techniques.
- Learn how to utilize the Universal Sentence Encoder to convert text into high-dimensional vector space representations.
- Gain practical experience with FAISS (Facebook AI Similarity Search), an efficient library for indexing and searching high-dimensional vectors.
- Apply these techniques to build a fully functioning semantic search engine that can interpret and respond to natural language queries.
- By accomplishing these objectives, you will acquire a comprehensive skill set that underpins advanced search functionalities in modern AI-driven systems, preparing you for further exploration and development in the field of natural language processing and information retrieval.
<p>
<h2>Setup</h2>
To ensure a smooth experience throughout this lab, we need to set up our environment properly. This includes installing necessary libraries, importing them, and preparing helper functions that will be used later in the lab.
<p>
<h2>Installing Required Libraries</h2>
<p>
Before we start, you need to install the following libraries if you haven't already:
<p>
- tensorflow: The core library for TensorFlow, required for working with the Universal Sentence Encoder.
- tensorflow-hub: A library that makes it easy to download and deploy pre-trained TensorFlow models, including the Universal Sentence Encoder.
- faiss-cpu: A library for efficient similarity search and clustering of dense vectors.
- numpy: A library for numerical computing, which we will use to handle arrays and matrices.
- scikit-learn: A machine learning library that provides various tools for data mining and data analysis, useful for additional tasks like data splitting and evaluation metrics.
<p>
You can install these libraries using pip with the following commands:

You will need to run the following cell to install them:

In [6]:
%pip install "setuptools<81"
!pip install faiss-cpu numpy scikit-learn
!pip install "tensorflow>=2.0.0"
!pip install --upgrade tensorflow-hub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
import faiss
import re
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from pprint import pprint

# Suppressing warnings
def warn(*args, **kwargs):
    pass

import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

d:\code\pythondemo\notebooks\agents\chroma\rag_with_vector_database\.rag_with_vector_database\Lib\site-packages\tensorflow_hub\__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


<h2>Understanding Semantic Search</h2>
<p>
When we're looking to build a semantic search engine, it's important to start with the basics. Let's break down what semantic search is and why it's a game-changer in finding information.

<h3>What is Semantic Search?</h3>
Semantic search transcends the limitations of traditional keyword searches by understanding the context and nuances of language in user queries. At its core, semantic search:

- Enhances the search experience by <u>interpreting the intent and contextual meaning behind search queries</u>.
- Delivers more accurate and relevant search results by <u>analyzing the relationships between words and phrases within the search context</u>.
- Adapts to <u>user behavior and preferences</u>, refining search results for better user satisfaction.

<h3>How Semantic Search Works - The Simple Version</h3>
Now, how does this smart assistant do its job? It uses some clever tricks from a field called Natural Language Processing, or NLP for short. Here’s the simple version of the process:

- Getting the Gist: First up, the search engine listens to your query and tries to get the gist of it. Instead of just spotting keywords, it digs deeper to <u>find the real meaning</u>.
- Making Connections: Next, it thinks about all the <u>different ways words can be related</u> (like "doctor" and "physician" meaning the same thing). This helps it get a better sense of what you're asking for.
- Picking the Best: Finally, it acts like a librarian who knows every book in the library. It sorts through tons of information to <u>pick what matches your query best, considering what you probably mean</u>.

<h3>The Technical Side of Semantic Search</h3>
After understanding the basics, let's peek under the hood at the technical engine powering semantic search. This part is a bit like math class, where we learn about vectors — no, not the ones you learned in physics, but something similar that we use in search engines.

<h4>Vectors: The Language of Semantic Search</h4>
In the world of semantic search, a <u>vector is a list of numbers that a computer uses to represent the meaning of words or sentences</u>. Imagine each word or sentence as a point in space. The closer two points are, the more similar their meanings.

- Creating Vectors: We start by turning words or <u>sentences into vectors using models like the Universal Sentence Encoder</u>. It's like giving each piece of text its unique numerical fingerprint.
- Calculating Similarity: To find out how similar two pieces of text are, we measure how close their vectors are in space. This is done using mathematical formulas, such as cosine similarity, which tells us how similar or different two text fingerprints are.
- Using Vectors for Search: When you search for something, the search engine looks for the vectors closest to the vector of your query. The closest vectors represent the most relevant results to what you're asking.

<h3>How Vectors Power Our Search</h3>
Vectors are powerful because they can capture the subtle meanings of language that go beyond the surface of words. Here's what happens in a semantic search engine:

- Vectorization: When we type in a search query, the engine immediately turns our words into a vector.
- Indexing: It then quickly scans through a massive index of other vectors, each representing different pieces of information.
- Retrieval: By finding the closest matching vectors, the engine retrieves information that's not just textually similar but semantically related.

By the end of this guide, you'll understand how to create a search engine that does all of this and more. We'll start simple and build up step by step. Ready? Let's get started!

<h2>Understanding Vectorization and Indexing</h2>
Vectorization and indexing are key components of building a semantic search engine. Let's explore how they work using the Universal Sentence Encoder (USE) and FAISS.

<h3>What does the Universal Sentence Encoder do?</h3>
The Universal Sentence Encoder (USE) takes sentences, no matter how complex, and turns them into vectors. These vectors are arrays of numbers that capture the essence of sentences. Here's why it's amazing:

- Language Comprehension: USE understands the meaning of sentences by considering the context in which each word is used.
- Versatility: It's trained on a variety of data sources, enabling it to handle a wide range of topics and sentence structures.
- Speed: Once trained, USE can quickly convert sentences to vectors, making it highly efficient.

<h3>How does the Universal Sentence Encoder work?</h3>
The magic of USE lies in its training. It uses deep learning models to digest vast amounts of text. Here’s what it does:

- Analyzes Words: It looks at each word in a sentence and the words around it to get a full picture of their meaning.
- Understands Context: It pays attention to the order of words and how they're used together to grasp the sentence's intent.
- Creates Vectors: It converts all this understanding into a numeric vector that represents the sentence.

<h3>What is FAISS and what does it do?</h3>
FAISS, developed by Facebook AI, is a library for efficient similarity search. After we have vectors from USE, we need a way to search through them quickly to find the most relevant ones to a query. FAISS does just that:

- Efficient Searching: It uses optimized algorithms to rapidly search through large collections of vectors.
- Scalability: It can handle databases of vectors that are too large to fit in memory, making it suitable for big data applications.
- Accuracy: It provides highly accurate search results, thanks to its advanced indexing strategies.

<h3>How does FAISS work?</h3>
FAISS creates an index of all the vectors, which allows it to search through them efficiently. Here's a simplified version of its process:

- Index Building: It organizes vectors in a way that similar ones are near each other, making it faster to find matches.
- Searching: When you search with a new vector, FAISS quickly identifies which part of the index to look at for the closest matches.
- Retrieving Results: It then retrieves the most similar vectors, which correspond to the most relevant search results.

<h3>Putting it all together:</h3>

With USE and FAISS, we have a powerful duo. USE helps us understand language in numerical terms, and FAISS lets us search through these numbers to find meaningful connections. Combining them, we create a semantic search engine that's both smart and swift.

<h2>The 20 Newsgroups Dataset</h2>
In this project, we'll be using the 20 Newsgroups dataset, a collection of approximately 20,000 newsgroup documents, partitioned across 20 different newsgroups. It's a go-to dataset in the NLP community because it presents real-world challenges:

<h2>What is the 20 Newsgroups Dataset?</h2>

- Diverse Topics: The dataset spans 20 different topics, from sports and science to politics and religion, reflecting the diverse interests of newsgroup members.
- Natural Language: It contains actual discussions, with all the nuances of human language, making it ideal for semantic search.
- Prevalence of Context: The conversations within it require understanding of context to differentiate between the topics effectively.

<h2>How are we using the 20 Newsgroups Dataset?</h2>

- Exploring Data: We'll start by loading the dataset and exploring its structure to understand the kind of information it holds.
- Preprocessing: We'll clean the text data, removing any unwanted noise that could affect our semantic analysis.
- Vectorization: We'll then use the Universal Sentence Encoder to transform this text into numerical vectors that capture the essence of each document.
- Semantic Search Implementation: Finally, we'll use FAISS to index these vectors, allowing us to perform fast and efficient semantic searches across the dataset.

By working with the 20 Newsgroups dataset, you'll gain hands-on experience with real-world data and the end-to-end process of building a semantic search engine.

In [16]:
import os
from sklearn.datasets import fetch_20newsgroups, load_files
dataset_zip = "D:\\.scikit_learn_data\\20newsbydate.tar.gz"
test_dataset_path = "D:\\.scikit_learn_data\\20news-bydate-test"
train_dataset_path = "D:\\.scikit_learn_data\\20news-bydate-train"

if not os.path.isfile(dataset_zip):
    print("Downloading Dataset ...works only on Linux/Mac.")
    newsgroups_train = fetch_20newsgroups(subset='train', data_home='D:\\.scikit_learn_data') # does not work on windows (?)
else:
    print("Dataset already exists. Skipping download.")
    newsgroups_train =  load_files(container_path=train_dataset_path, encoding='latin1')
    news_groups_test = load_files(container_path=test_dataset_path, encoding='latin1')

Dataset already exists. Skipping download.


In [13]:
pprint(list(newsgroups_train.target_names))

['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']


In [14]:
# Display the first 3 posts from the dataset
for i in range(3):
    print(f"Sample post {i+1}:\n")
    pprint(newsgroups_train.data[i])
    print("\n" + "-"*80 + "\n")

Sample post 1:

('From: cubbie@garnet.berkeley.edu (                               )\r\n'
 'Subject: Re: Cubs behind Marlins? How?\r\n'
 'Article-I.D.: agate.1pt592$f9a\r\n'
 'Organization: University of California, Berkeley\r\n'
 'Lines: 12\r\n'
 'NNTP-Posting-Host: garnet.berkeley.edu\r\n'
 '\r\n'
 '\r\n'
 'gajarsky@pilot.njin.net writes:\r\n'
 '\r\n'
 "morgan and guzman will have era's 1 run higher than last year, and\r\n"
 ' the cubs will be idiots and not pitch harkey as much as hibbard.\r\n'
 " castillo won't be good (i think he's a stud pitcher)\r\n"
 '\r\n'
 '       This season so far, Morgan and Guzman helped to lead the Cubs\r\n'
 '       at top in ERA, even better than THE rotation at Atlanta.\r\n'
 '       Cubs ERA at 0.056 while Braves at 0.059. We know it is early\r\n'
 '       in the season, we Cubs fans have learned how to enjoy the\r\n'
 '       short triumph while it is still there.\r\n')

-------------------------------------------------------------------------------

<h2>Pre-processing Data</h2>
In this section, we focus on preparing the text data from the 20 Newsgroups dataset for our semantic search engine. Preprocessing is a critical step to ensure the quality and consistency of the data before it's fed into the Universal Sentence Encoder.

<h3>Steps in Preprocessing:</h3>

<h4>Fetching Data:</h4>

We load the complete 20 Newsgroups dataset using fetch_20newsgroups from sklearn.datasets.
documents = newsgroups.data stores all the newsgroup documents in a list.

<h4>Defining the Preprocessing Function:</h4>

The preprocess_text function is designed to clean each text document. Here's what it does to every piece of text:

- Removes Email Headers: Strips off lines that start with 'From:' as they usually contain metadata like email addresses.
- Eliminates Email Addresses: Finds patterns resembling email addresses and removes them.
- Strips Punctuations and Numbers: Removes all characters except alphabets, aiding in focusing on textual data.
- Converts to Lowercase: Standardizes the text by converting all characters to lowercase, ensuring uniformity.
- Trims Excess Whitespace: Cleans up any extra spaces, tabs, or line breaks.

<h4>Applying Preprocessing:</h4>

- We iterate over each document in the documents list and apply our preprocess_text function.
- The cleaned documents are stored in processed_documents, ready for further processing.

By preprocessing the text data in this way, we reduce noise and standardize the text, which is essential for achieving meaningful semantic analysis in later steps.

In [19]:
newsgroups = fetch_20newsgroups(subset='all', data_home='D:\\.scikit_learn_data')
documents = newsgroups.data

# Basic preprocessing of text data
def preprocess_text(text):
    # Remove email headers
    text = re.sub(r'^From:.*\n?', '', text, flags=re.MULTILINE)
    # Remove email addresses
    text = re.sub(r'\S*@\S*\s?', '', text)
    # Remove punctuations and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    # Remove excess whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Preprocess each document
processed_documents = [preprocess_text(doc) for doc in documents]

In [20]:
# Choose a sample post to display
sample_index = 0  # for example, the first post in the dataset

# Print the original post
print("Original post:\n")
print(newsgroups_train.data[sample_index])
print("\n" + "-"*80 + "\n")

# Print the preprocessed post
print("Preprocessed post:\n")
print(preprocess_text(newsgroups_train.data[sample_index]))
print("\n" + "-"*80 + "\n")

Original post:

From: cubbie@garnet.berkeley.edu (                               )
Subject: Re: Cubs behind Marlins? How?
Article-I.D.: agate.1pt592$f9a
Organization: University of California, Berkeley
Lines: 12
NNTP-Posting-Host: garnet.berkeley.edu


gajarsky@pilot.njin.net writes:

morgan and guzman will have era's 1 run higher than last year, and
 the cubs will be idiots and not pitch harkey as much as hibbard.
 castillo won't be good (i think he's a stud pitcher)

       This season so far, Morgan and Guzman helped to lead the Cubs
       at top in ERA, even better than THE rotation at Atlanta.
       Cubs ERA at 0.056 while Braves at 0.059. We know it is early
       in the season, we Cubs fans have learned how to enjoy the
       short triumph while it is still there.


--------------------------------------------------------------------------------

Preprocessed post:

subject re cubs behind marlins how articleid agateptfa organization university of california berkeley lines nn

<h2>Universal Sentence Encoder</h2>

After preprocessing the text data, the next step is to transform this cleaned text into numerical vectors using the Universal Sentence Encoder (USE). These vectors capture the semantic essence of the text.

<h3>Loading the USE Module:</h3>

We use TensorFlow Hub (hub) to load the pre-trained Universal Sentence Encoder.

embed = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4") fetches the USE module, making it ready for vectorization.

<h3>Defining the Embedding Function:</h3>

The embed_text function is defined to take a piece of text as input and return its vector representation.

Inside the function, embed(text) converts the text into a high-dimensional vector, capturing the nuanced semantic meaning.

.numpy() is used to convert the result from a TensorFlow tensor to a NumPy array, which is a more versatile format for subsequent operations.

<h3>Vectorizing Preprocessed Documents:</h3>

We then apply the embed_text function to each document in our preprocessed dataset, processed_documents.

np.vstack([...]) stacks the vectors vertically to create a 2D array, where each row represents a document.

The resulting array X_use holds the vectorized representations of all the preprocessed documents, ready to be used for semantic search indexing and querying.

By vectorizing the text with USE, we've now converted our textual data into a format that can be efficiently processed by machine learning algorithms, setting the stage for the next step: indexing with FAISS.

In [21]:
# Load the Universal Sentence Encoder's TF Hub module
embed = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")

# Function to generate embeddings
def embed_text(text):
    return embed(text).numpy()

# Generate embeddings for each preprocessed document
X_use = np.vstack([embed_text([doc]) for doc in processed_documents])

<h2>Indexing with FAISS</h2>

With our documents now represented as vectors using the Universal Sentence Encoder, the next step is to use FAISS (Facebook AI Similarity Search) for efficient similarity searching.

<h3>Creating a FAISS Index</h3>

We first determine the dimension of our vectors from X_use using X_use.shape[1].

A FAISS index (index) is created specifically for L2 distance (Euclidean distance) using faiss.IndexFlatL2(dimension).

We add our document vectors to this index with index.add(X_use). This step effectively creates a searchable space for our document vectors.

<h3>Choosing the Right Index:</h3>
In this project, we use IndexFlatL2 for its simplicity and effectiveness in handling small to medium-sized datasets.
FAISS offers a variety of indexes tailored for different use cases and dataset sizes. Depending on your specific needs and the complexity of your data, you might consider other indexes for more efficient searching.
For larger datasets or more advanced use cases, indexes like IndexIVFFlat, IndexIVFPQ, and others can provide faster search times and reduced memory usage. Explore more at <a href="https://github.com/facebookresearch/faiss/wiki/Faiss-indexes">FAISS indexes wiki.</a>

In [22]:
dimension = X_use.shape[1]
index = faiss.IndexFlatL2(dimension)  # Creating a FAISS index
index.add(X_use)  # Adding the document vectors to the index

<h2>Quering with FAISS</h2>

<h3>Defining the Search Function:</h3>

- The search function is designed to find documents that are semantically similar to a given query.
- It preprocesses the query text using the preprocess_text function to ensure consistency.
- The query text is then converted to a vector using embed_text.
- FAISS performs a search for the nearest neighbors (k) to this query vector in our index.
- It returns the distances and indices of these nearest neighbors.

<h3>Executing a Query and Displaying Results:</h3>

 - We test our search engine with an example query (e.g., "motorcycle").
- The search function returns the indices of the documents in the index that are most similar to the query.
 - For each result, we display:
    - The ranking of the result (based on distance).
    - The distance value itself, indicating how close the document is to the query.
    - The actual text of the document. We display both the preprocessed and original versions of each document for comparison.
    
This functionality showcases the practical application of semantic search: retrieving information that is contextually relevant to the query, not just based on keyword matching. The displayed results will give a clear idea of how our semantic search engine interprets and responds to natural language queries.

In [23]:
# Function to perform a query using the Faiss index
def search(query_text, k=5):
    # Preprocess the query text
    preprocessed_query = preprocess_text(query_text)
    # Generate the query vector
    query_vector = embed_text([preprocessed_query])
    # Perform the search
    distances, indices = index.search(query_vector.astype('float32'), k)
    return distances, indices

# Example Query
query_text = "motorcycle"
distances, indices = search(query_text)

# Display the results
for i, idx in enumerate(indices[0]):
    # Ensure that the displayed document is the preprocessed one
    print(f"Rank {i+1}: (Distance: {distances[0][i]})\n{processed_documents[idx]}\n")

Rank 1: (Distance: 1.0367169380187988)
subject first bike organization freshman mechanical engineering carnegie mellon pittsburgh pa lines nntppostinghost andrewcmuedu anyone i am a serious motorcycle enthusiast without a motorcycle and to put it bluntly it sucks i really would like some advice on what would be a good starter bike for me i do know one thing however i need to make my first bike a good one because buying a second any time soon is out of the question i am specifically interested in racing bikes cbr f gsxr i know that this may sound kind of crazy considering that ive never had a bike before but i am responsible a fast learner and in love please give me any advice that you think would help me in my search including places to look or even specific bikes that you want to sell me thanks jamie belliveau

Rank 2: (Distance: 1.0392436981201172)
subject first bike organization freshman mechanical engineering carnegie mellon pittsburgh pa lines nntppostinghost poandrewcmuedu anyone

In [24]:
# Display the results
for i, idx in enumerate(indices[0]):
    # Displaying the original (unprocessed) document corresponding to the search result
    print(f"Rank {i+1}: (Distance: {distances[0][i]})\n{documents[idx]}\n")

Rank 1: (Distance: 1.0367169380187988)
From: James Leo Belliveau <jbc9+@andrew.cmu.edu>
Subject: First Bike??
Organization: Freshman, Mechanical Engineering, Carnegie Mellon, Pittsburgh, PA
Lines: 17
NNTP-Posting-Host: andrew.cmu.edu

 Anyone, 

    I am a serious motorcycle enthusiast without a motorcycle, and to
put it bluntly, it sucks.  I really would like some advice on what would
be a good starter bike for me.  I do know one thing however, I need to
make my first bike a good one, because buying a second any time soon is
out of the question.  I am specifically interested in racing bikes, (CBR
600 F2, GSX-R 750).  I know that this may sound kind of crazy
considering that I've never had a bike before, but I am responsible, a
fast learner, and in love.  Please give me any advice that you think
would help me in my search, including places to look or even specific
bikes that you want to sell me.

    Thanks  :-)

    Jamie Belliveau (jbc9@andrew.cmu.edu)  



Rank 2: (Distance: 1.03924